# Build Streamflow Training Parquet V2

This notebook replaces the original `build_streamflow_training_parquet.ipynb` with a cleaner forecasting dataset for **multi-step discharge prediction**.

Design used here:
- One Parquet file per stream site.
- Input history: `168` hourly discharge values from the target site.
- Input context: average air temperature over the last `24` hours from climate site `2`.
- Input context: average precipitation over the last `24` hours from climate site `2`, with only limited short-gap filling.
- Static/current context per sample: latest snow depth from climate site `1`.
- Calendar context per sample: `year`, `month`, `day`, `hour`, `day_of_week`, `day_of_year`, `season`.
- Targets: next `24` hourly discharge values.

The output schema is intentionally fixed and model-friendly so it can be used for Temporal Fusion Transformer or other sequence forecasting models after reshaping.


## Review of Issues in the Original Notebook

The original notebook has some useful ideas, especially hourly aggregation and gap filling, but it does not match the forecasting problem you described.

Main issues:
- It builds a **next-hour** target only, not a **24-hour horizon**.
- It auto-discovers **every variable** in `public.variable` and uses all of them as features. That makes the schema unstable and harder to reproduce.
- Duplicate variable names are resolved by appending database IDs, which means feature columns can change if the database changes.
- Negative values are converted to missing values for **all variables**. That is wrong for variables like air temperature, where negative readings are physically valid.
- Snow depth is not explicitly pinned to the climate station you described, so the selected source site can drift if the availability logic changes.
- The imputation strategy is the same for all features, but snow depth is better handled as a last-observed value carried forward instead of full time interpolation.
- The notebook outputs one wide file for one target site instead of generating a standard per-site discharge forecasting dataset.
- Naming is inconsistent: the database variable is `Discharge`, but the old notebook renames it to `streamflow`. That is not fatal, but it is confusing.

This replacement notebook keeps the good parts and fixes the forecasting shape, feature selection, target construction, and variable-specific cleaning choices.

## 1. Configure Database and Dataset Settings

In [4]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError as exc:
    raise ImportError(
        "This notebook needs PyArrow. Install it with: pip install pyarrow"
    ) from exc

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5433")
DB_NAME = os.getenv("DB_NAME", "database")
DB_USER = os.getenv("DB_USER", "admin")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

TARGET_SITE_IDS = [3, 4]
DISCHARGE_SOURCE_IS_TARGET_SITE = True
AIR_TEMP_SITE_ID = 2
PRECIPITATION_SITE_ID = 2
SNOW_DEPTH_SITE_ID = 1

LOOKBACK_HOURS = 168
AIR_TEMP_LOOKBACK_HOURS = 24
FORECAST_HORIZON_HOURS = 24
HOURLY_BINNING = "hour_end"
MAX_CONSECUTIVE_MISSING_HOURS = 10
DEFAULT_SAMPLE_STRIDE_HOURS = 6
HIGH_DENSITY_MONTHS = {3, 4, 5, 6}
HIGH_DENSITY_SAMPLE_STRIDE_HOURS = 1

OUTPUT_DIR = Path("streamflow_parquet_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SITE_OUTPUT_TEMPLATE = "discharge_training_site_{site_id}_lb168_air24avg_precip24avg_h24.parquet"

DISCHARGE_MATCH_TERMS = ["discharge", "streamflow", "stream flow"]
AIR_TEMP_MATCH_TERMS = ["airtemp", "air temp", "air temperature"]
PRECIPITATION_MATCH_TERMS = ["precip", "precipitation"]
SNOW_DEPTH_MATCH_TERMS = ["snowdepth", "snow depth"]



## 2. Inspect Available Sites and Variables

In [10]:
sites_df = pd.read_sql_query(
    text("""
    SELECT site_id, site_code, name, site_type, state, county
    FROM public.site
    ORDER BY site_id
    """),
    engine,
)

variables_df = pd.read_sql_query(
    text("""
    SELECT
        v.variable_id,
        v.variable_code,
        v.name AS variable_name,
        v.variable_type,
        u.name AS unit_name,
        u.symbol AS unit_symbol
    FROM public.variable v
    LEFT JOIN public.unit u ON v.unit_id = u.unit_id
    ORDER BY v.variable_id
    """),
    engine,
)

availability_df = pd.read_sql_query(
    text("""
    SELECT
        s.site_id,
        s.site_code,
        v.variable_id,
        v.variable_code,
        v.name AS variable_name,
        COUNT(*) AS row_count,
        MIN(d.datetime_utc) AS min_datetime,
        MAX(d.datetime_utc) AS max_datetime
    FROM public.datastream d
    JOIN public.site s ON d.site_id = s.site_id
    JOIN public.variable v ON d.variable_id = v.variable_id
    GROUP BY s.site_id, s.site_code, v.variable_id, v.variable_code, v.name
    ORDER BY s.site_id, v.variable_id
    """),
    engine,
)

display(sites_df)
display(variables_df)
display(availability_df)

,site_id,site_code,name,site_type,state,county
0,1,LR_TG_C,Climate Station at Tony Grove,Atmosphere,Utah,Cache
1,2,LR_GC_C,Climate Station at Logan River Golf Course,Atmosphere,Utah,Cache
2,3,LR_WaterLab_AA,Logan River at the Utah Water Research Laboratory west bridge,Stream,Utah,Cache
3,4,LR_Mendon_AA,Logan River at Mendon Road (600 South),Stream,Utah,Cache


,variable_id,variable_code,variable_name,variable_type,unit_name,unit_symbol
0,1,SnowDepth,Snow depth,Hydrology,centimeter,cm
1,2,WaterTemp,Water Temperature,Water Quality,degree Celsius,C
2,3,ODO,"Oxygen, dissolved",Water Quality,milligrams per liter,mg/L
3,4,Discharge,Discharge,Hydrology,cubic feet per second,cfs
4,5,AirTemp,Air Temperature,Climate,degree Celsius,C
5,6,Precip,Precipitation,Hydrology,centimeter,cm


,site_id,site_code,variable_id,variable_code,variable_name,row_count,min_datetime,max_datetime
0,1,LR_TG_C,1,SnowDepth,Snow depth,423331,2014-02-19 21:15:00,2026-03-24 17:00:00
1,1,LR_TG_C,5,AirTemp,Air Temperature,320421,2017-02-01 19:45:00,2026-03-24 17:00:00
2,2,LR_GC_C,5,AirTemp,Air Temperature,321544,2017-01-20 23:00:00,2026-03-24 18:00:00
3,2,LR_GC_C,6,Precip,Precipitation,838768,2014-03-18 15:30:00,2026-03-24 17:00:00
4,3,LR_WaterLab_AA,2,WaterTemp,Water Temperature,435460,2013-10-02 20:15:00,2026-03-24 19:00:00
5,3,LR_WaterLab_AA,3,ODO,"Oxygen, dissolved",435148,2013-10-02 20:15:00,2026-03-24 18:00:00
6,3,LR_WaterLab_AA,4,Discharge,Discharge,428202,2014-01-01 07:00:00,2026-03-24 06:00:00
7,4,LR_Mendon_AA,2,WaterTemp,Water Temperature,436398,2013-10-11 22:15:00,2026-03-24 19:00:00
8,4,LR_Mendon_AA,4,Discharge,Discharge,428591,2014-01-01 07:15:00,2026-03-24 19:00:00


## 3. Helper Functions

In [14]:
def month_to_season(month):
    if month in (12, 1, 2):
        return 0
    if month in (3, 4, 5):
        return 1
    if month in (6, 7, 8):
        return 2
    return 3


def find_variable_ids(variables, match_terms):
    searchable = (
        variables["variable_code"].fillna("") + " " + variables["variable_name"].fillna("")
    ).str.lower()
    mask = pd.Series(False, index=variables.index)
    for term in match_terms:
        mask = mask | searchable.str.contains(term.lower(), regex=False)

    matched = variables.loc[mask].copy()
    if matched.empty:
        raise ValueError(f"Could not match any variable for terms: {match_terms}")
    return matched


def assign_hourly_bucket(datetime_series, binning="hour_end"):
    if binning == "hour_end":
        return datetime_series.dt.ceil("h")
    if binning == "hour_start":
        return datetime_series.dt.floor("h")
    raise ValueError("HOURLY_BINNING must be hour_end or hour_start.")


def load_raw_feature(engine, site_id, variable_ids, feature_name, allow_negative=True):
    query = text("""
        SELECT
            d.site_id,
            d.variable_id,
            d.datetime_utc AS datetime,
            d.value
        FROM public.datastream d
        WHERE d.site_id = :site_id
          AND d.variable_id = ANY(:variable_ids)
        ORDER BY d.datetime_utc, d.variable_id
    """)
    df = pd.read_sql_query(
        query,
        engine,
        params={"site_id": int(site_id), "variable_ids": [int(v) for v in variable_ids]},
    )
    if df.empty:
        raise ValueError(f"No rows found for {feature_name} at site_id={site_id}")

    df["datetime"] = pd.to_datetime(df["datetime"])

    if not allow_negative:
        negative_count = int((df["value"] < 0).sum())
        if negative_count:
            df.loc[df["value"] < 0, "value"] = np.nan
            print(f"Converted {negative_count:,} negative {feature_name} values to missing")

    return df


def to_hourly_last_value(df, feature_name, binning="hour_end"):
    working = df.copy()
    working = working.dropna(subset=["value"]).sort_values("datetime")
    working["hourly_datetime"] = assign_hourly_bucket(working["datetime"], binning=binning)

    hourly = (
        working.groupby("hourly_datetime", as_index=False)
        .tail(1)[["hourly_datetime", "value"]]
        .rename(columns={"hourly_datetime": "datetime", "value": feature_name})
        .sort_values("datetime")
        .reset_index(drop=True)
    )
    return hourly


def add_calendar_features(df):
    out = df.copy()
    out["year"] = out["datetime"].dt.year
    out["month"] = out["datetime"].dt.month
    out["day"] = out["datetime"].dt.day
    out["hour"] = out["datetime"].dt.hour
    out["day_of_week"] = out["datetime"].dt.dayofweek
    out["day_of_year"] = out["datetime"].dt.dayofyear
    out["season"] = out["month"].map(month_to_season)
    return out


def has_consecutive_missing_gap(missing_series, min_gap_hours):
    missing = missing_series.fillna(False).astype(bool)
    run_length = 0
    for is_missing in missing:
        if is_missing:
            run_length += 1
            if run_length >= min_gap_hours:
                return True
        else:
            run_length = 0
    return False


def sample_stride_hours_for_month(month):
    if month in HIGH_DENSITY_MONTHS:
        return HIGH_DENSITY_SAMPLE_STRIDE_HOURS
    return DEFAULT_SAMPLE_STRIDE_HOURS


def write_parquet_with_pyarrow(df, output_path):
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, output_path)


def read_parquet_with_pyarrow(input_path):
    return pq.read_table(input_path).to_pandas()



In [15]:
discharge_variables = find_variable_ids(variables_df, DISCHARGE_MATCH_TERMS)
air_temp_variables = find_variable_ids(variables_df, AIR_TEMP_MATCH_TERMS)
precipitation_variables = find_variable_ids(variables_df, PRECIPITATION_MATCH_TERMS)
snow_depth_variables = find_variable_ids(variables_df, SNOW_DEPTH_MATCH_TERMS)

print("Discharge variable matches")
display(discharge_variables)

print("Air temperature variable matches")
display(air_temp_variables)

print("Precipitation variable matches")
display(precipitation_variables)

print("Snow depth variable matches")
display(snow_depth_variables)

Discharge variable matches


,variable_id,variable_code,variable_name,variable_type,unit_name,unit_symbol
3,4,Discharge,Discharge,Hydrology,cubic feet per second,cfs


Air temperature variable matches


,variable_id,variable_code,variable_name,variable_type,unit_name,unit_symbol
4,5,AirTemp,Air Temperature,Climate,degree Celsius,C


Precipitation variable matches


,variable_id,variable_code,variable_name,variable_type,unit_name,unit_symbol
5,6,Precip,Precipitation,Hydrology,centimeter,cm


Snow depth variable matches


,variable_id,variable_code,variable_name,variable_type,unit_name,unit_symbol
0,1,SnowDepth,Snow depth,Hydrology,centimeter,cm


## 4. Build Hourly Feature Table

In [12]:
def build_hourly_feature_table(
    engine,
    target_site_id,
    discharge_variable_ids,
    air_temp_variable_ids,
    snow_depth_variable_ids,
    precipitation_variable_ids,
    air_temp_site_id,
    snow_depth_site_id,
    precipitation_site_id,
    hourly_binning="hour_end",
):
    discharge_raw = load_raw_feature(
        engine,
        site_id=target_site_id,
        variable_ids=discharge_variable_ids,
        feature_name="discharge",
        allow_negative=False,
    )
    air_temp_raw = load_raw_feature(
        engine,
        site_id=air_temp_site_id,
        variable_ids=air_temp_variable_ids,
        feature_name="air_temp",
        allow_negative=True,
    )
    snow_depth_raw = load_raw_feature(
        engine,
        site_id=snow_depth_site_id,
        variable_ids=snow_depth_variable_ids,
        feature_name="snow_depth",
        allow_negative=False,
    )
    precipitation_raw = load_raw_feature(
        engine,
        site_id=precipitation_site_id,
        variable_ids=precipitation_variable_ids,
        feature_name="precipitation",
        allow_negative=False,
    )

    discharge_hourly = to_hourly_last_value(discharge_raw, "discharge", binning=hourly_binning)
    air_temp_hourly = to_hourly_last_value(air_temp_raw, "air_temp", binning=hourly_binning)
    snow_depth_hourly = to_hourly_last_value(snow_depth_raw, "snow_depth", binning=hourly_binning)
    precipitation_hourly = to_hourly_last_value(
        precipitation_raw,
        "precipitation",
        binning=hourly_binning,
    )

    min_dt = max(
        discharge_hourly["datetime"].min(),
        air_temp_hourly["datetime"].min(),
        snow_depth_hourly["datetime"].min(),
        precipitation_hourly["datetime"].min(),
    )
    max_dt = min(
        discharge_hourly["datetime"].max(),
        air_temp_hourly["datetime"].max(),
        snow_depth_hourly["datetime"].max(),
        precipitation_hourly["datetime"].max(),
    )

    hourly_index = pd.date_range(min_dt, max_dt, freq="h")
    hourly = pd.DataFrame({"datetime": hourly_index})
    hourly["site_id"] = int(target_site_id)

    hourly = hourly.merge(discharge_hourly, on="datetime", how="left")
    hourly = hourly.merge(air_temp_hourly, on="datetime", how="left")
    hourly = hourly.merge(snow_depth_hourly, on="datetime", how="left")
    hourly = hourly.merge(precipitation_hourly, on="datetime", how="left")

    hourly = hourly.sort_values("datetime").reset_index(drop=True)
    hourly["discharge_observed"] = hourly["discharge"]
    hourly["discharge_missing_raw"] = hourly["discharge"].isna()
    hourly["air_temp_missing_raw"] = hourly["air_temp"].isna()
    hourly["snow_depth_missing_raw"] = hourly["snow_depth"].isna()
    hourly["precipitation_missing_raw"] = hourly["precipitation"].isna()

    hourly = hourly.set_index("datetime")

    hourly["discharge"] = hourly["discharge"].interpolate(method="time").ffill().bfill()
    hourly["air_temp"] = hourly["air_temp"].interpolate(method="time").ffill().bfill()
    hourly["snow_depth"] = hourly["snow_depth"].ffill().bfill()

    # Precipitation is often sparse, so only fill short gaps rather than fully smoothing long missing runs.
    hourly["precipitation"] = (
        hourly["precipitation"]
        .interpolate(method="time", limit=3)
        .ffill(limit=3)
        .bfill(limit=3)
    )

    hourly["precip_avg_last_24h"] = hourly["precipitation"].rolling(window=24, min_periods=18).mean()

    hourly = hourly.reset_index()
    hourly = add_calendar_features(hourly)
    return hourly



## 5. Build Supervised Samples With 24-Hour Targets

In [13]:
def build_multihorizon_samples(
    hourly_df,
    discharge_lookback_hours=168,
    air_temp_lookback_hours=24,
    forecast_horizon_hours=24,
    max_consecutive_missing_hours=10,
):
    rows = []
    expected_step = pd.Timedelta(hours=1)
    raw_missing_columns = [
        "discharge_missing_raw",
        "air_temp_missing_raw",
        "snow_depth_missing_raw",
        "precipitation_missing_raw",
    ]

    hourly_df = hourly_df.sort_values(["site_id", "datetime"]).reset_index(drop=True)

    for site_id, site_df in hourly_df.groupby("site_id", sort=True):
        site_df = site_df.sort_values("datetime").reset_index(drop=True)
        last_start = len(site_df) - forecast_horizon_hours + 1
        target_start_idx = discharge_lookback_hours

        while target_start_idx < last_start:
            history_window = site_df.iloc[target_start_idx - discharge_lookback_hours:target_start_idx]
            air_window = site_df.iloc[target_start_idx - air_temp_lookback_hours:target_start_idx]
            precip_window = site_df.iloc[target_start_idx - 24:target_start_idx]
            future_window = site_df.iloc[target_start_idx:target_start_idx + forecast_horizon_hours]

            forecast_month = int(future_window.iloc[0]["month"])
            stride_hours = sample_stride_hours_for_month(forecast_month)

            full_window = site_df.iloc[
                target_start_idx - discharge_lookback_hours:target_start_idx + forecast_horizon_hours
            ]
            full_time_window = full_window["datetime"]

            if not (full_time_window.diff().dropna() == expected_step).all():
                target_start_idx += stride_hours
                continue

            if len(air_window) != air_temp_lookback_hours or len(precip_window) != 24:
                target_start_idx += stride_hours
                continue

            if future_window["discharge_observed"].isna().any():
                target_start_idx += stride_hours
                continue

            if any(
                has_consecutive_missing_gap(full_window[column], max_consecutive_missing_hours)
                for column in raw_missing_columns
            ):
                target_start_idx += stride_hours
                continue

            precip_avg_last_24h = precip_window["precip_avg_last_24h"].iloc[-1]
            if pd.isna(precip_avg_last_24h):
                target_start_idx += stride_hours
                continue

            sample = {
                "site_id": int(site_id),
                "history_end_time": history_window.iloc[-1]["datetime"],
                "forecast_start_time": future_window.iloc[0]["datetime"],
                "forecast_end_time": future_window.iloc[-1]["datetime"],
                "snow_depth_latest": float(history_window.iloc[-1]["snow_depth"]),
                "air_temp_avg_last_24h": float(air_window["air_temp"].mean()),
                "precip_avg_last_24h": float(precip_avg_last_24h),
                "year": int(future_window.iloc[0]["year"]),
                "month": int(future_window.iloc[0]["month"]),
                "day": int(future_window.iloc[0]["day"]),
                "hour": int(future_window.iloc[0]["hour"]),
                "day_of_week": int(future_window.iloc[0]["day_of_week"]),
                "day_of_year": int(future_window.iloc[0]["day_of_year"]),
                "season": int(future_window.iloc[0]["season"]),
                "sample_stride_hours": int(stride_hours),
            }

            for offset in range(discharge_lookback_hours, 0, -1):
                value = history_window.iloc[discharge_lookback_hours - offset]["discharge"]
                sample[f"discharge_t-{offset}"] = float(value)

            for step in range(1, forecast_horizon_hours + 1):
                value = future_window.iloc[step - 1]["discharge_observed"]
                sample[f"target_discharge_t+{step}"] = float(value)

            rows.append(sample)
            target_start_idx += stride_hours

    return pd.DataFrame(rows)


def build_training_parquet_for_site(
    engine,
    target_site_id,
    discharge_variable_ids,
    air_temp_variable_ids,
    snow_depth_variable_ids,
    precipitation_variable_ids,
    air_temp_site_id,
    snow_depth_site_id,
    precipitation_site_id,
    output_path,
    discharge_lookback_hours=168,
    air_temp_lookback_hours=24,
    forecast_horizon_hours=24,
    hourly_binning="hour_end",
    max_consecutive_missing_hours=10,
):
    hourly = build_hourly_feature_table(
        engine=engine,
        target_site_id=target_site_id,
        discharge_variable_ids=discharge_variable_ids,
        air_temp_variable_ids=air_temp_variable_ids,
        snow_depth_variable_ids=snow_depth_variable_ids,
        precipitation_variable_ids=precipitation_variable_ids,
        air_temp_site_id=air_temp_site_id,
        snow_depth_site_id=snow_depth_site_id,
        precipitation_site_id=precipitation_site_id,
        hourly_binning=hourly_binning,
    )

    training = build_multihorizon_samples(
        hourly,
        discharge_lookback_hours=discharge_lookback_hours,
        air_temp_lookback_hours=air_temp_lookback_hours,
        forecast_horizon_hours=forecast_horizon_hours,
        max_consecutive_missing_hours=max_consecutive_missing_hours,
    )

    if training.empty:
        raise ValueError(f"No training samples created for site_id={target_site_id}")

    output_path = Path(output_path)
    write_parquet_with_pyarrow(training, output_path)
    print(f"Saved {len(training):,} rows and {training.shape[1]:,} columns to {output_path}")
    return training, hourly



## 6. Build One Parquet File Per Stream Site

In [14]:
site_outputs = {}
site_hourly_frames = {}

for target_site_id in TARGET_SITE_IDS:
    output_path = OUTPUT_DIR / SITE_OUTPUT_TEMPLATE.format(site_id=target_site_id)
    training_df, hourly_df = build_training_parquet_for_site(
        engine=engine,
        target_site_id=target_site_id,
        discharge_variable_ids=discharge_variables["variable_id"].tolist(),
        air_temp_variable_ids=air_temp_variables["variable_id"].tolist(),
        snow_depth_variable_ids=snow_depth_variables["variable_id"].tolist(),
        precipitation_variable_ids=precipitation_variables["variable_id"].tolist(),
        air_temp_site_id=AIR_TEMP_SITE_ID,
        snow_depth_site_id=SNOW_DEPTH_SITE_ID,
        precipitation_site_id=PRECIPITATION_SITE_ID,
        output_path=output_path,
        discharge_lookback_hours=LOOKBACK_HOURS,
        air_temp_lookback_hours=AIR_TEMP_LOOKBACK_HOURS,
        forecast_horizon_hours=FORECAST_HORIZON_HOURS,
        hourly_binning=HOURLY_BINNING,
        max_consecutive_missing_hours=MAX_CONSECUTIVE_MISSING_HOURS,
    )
    site_outputs[target_site_id] = training_df
    site_hourly_frames[target_site_id] = hourly_df

    print(f"\nPreview for site_id={target_site_id}")
    display(training_df.head())



Converted 11,900 negative discharge values to missing
Converted 13,990 negative snow_depth values to missing
Converted 208 negative precipitation values to missing
Saved 33,108 rows and 207 columns to streamflow_parquet_v2/discharge_training_site_3_lb168_air24avg_precip24avg_h24.parquet

Preview for site_id=3


,site_id,history_end_time,forecast_start_time,forecast_end_time,snow_depth_latest,air_temp_avg_last_24h,precip_avg_last_24h,year,month,day,hour,day_of_week,day_of_year,season,sample_stride_hours,discharge_t-168,discharge_t-167,discharge_t-166,discharge_t-165,discharge_t-164,discharge_t-163,discharge_t-162,discharge_t-161,discharge_t-160,discharge_t-159,discharge_t-158,discharge_t-157,discharge_t-156,discharge_t-155,discharge_t-154,discharge_t-153,discharge_t-152,discharge_t-151,discharge_t-150,discharge_t-149,discharge_t-148,discharge_t-147,discharge_t-146,discharge_t-145,discharge_t-144,discharge_t-143,discharge_t-142,discharge_t-141,discharge_t-140,discharge_t-139,discharge_t-138,discharge_t-137,discharge_t-136,discharge_t-135,discharge_t-134,discharge_t-133,discharge_t-132,discharge_t-131,discharge_t-130,discharge_t-129,discharge_t-128,discharge_t-127,discharge_t-126,discharge_t-125,discharge_t-124,discharge_t-123,discharge_t-122,discharge_t-121,discharge_t-120,discharge_t-119,discharge_t-118,discharge_t-117,discharge_t-116,discharge_t-115,discharge_t-114,discharge_t-113,discharge_t-112,discharge_t-111,discharge_t-110,discharge_t-109,discharge_t-108,discharge_t-107,discharge_t-106,discharge_t-105,discharge_t-104,discharge_t-103,discharge_t-102,discharge_t-101,discharge_t-100,discharge_t-99,discharge_t-98,discharge_t-97,discharge_t-96,discharge_t-95,discharge_t-94,discharge_t-93,discharge_t-92,discharge_t-91,discharge_t-90,discharge_t-89,discharge_t-88,discharge_t-87,discharge_t-86,discharge_t-85,discharge_t-84,...,discharge_t-76,discharge_t-75,discharge_t-74,discharge_t-73,discharge_t-72,discharge_t-71,discharge_t-70,discharge_t-69,discharge_t-68,discharge_t-67,discharge_t-66,discharge_t-65,discharge_t-64,discharge_t-63,discharge_t-62,discharge_t-61,discharge_t-60,discharge_t-59,discharge_t-58,discharge_t-57,discharge_t-56,discharge_t-55,discharge_t-54,discharge_t-53,discharge_t-52,discharge_t-51,discharge_t-50,discharge_t-49,discharge_t-48,discharge_t-47,discharge_t-46,discharge_t-45,discharge_t-44,discharge_t-43,discharge_t-42,discharge_t-41,discharge_t-40,discharge_t-39,discharge_t-38,discharge_t-37,discharge_t-36,discharge_t-35,discharge_t-34,discharge_t-33,discharge_t-32,discharge_t-31,discharge_t-30,discharge_t-29,discharge_t-28,discharge_t-27,discharge_t-26,discharge_t-25,discharge_t-24,discharge_t-23,discharge_t-22,discharge_t-21,discharge_t-20,discharge_t-19,discharge_t-18,discharge_t-17,discharge_t-16,discharge_t-15,discharge_t-14,discharge_t-13,discharge_t-12,discharge_t-11,discharge_t-10,discharge_t-9,discharge_t-8,discharge_t-7,discharge_t-6,discharge_t-5,discharge_t-4,discharge_t-3,discharge_t-2,discharge_t-1,target_discharge_t+1,target_discharge_t+2,target_discharge_t+3,target_discharge_t+4,target_discharge_t+5,target_discharge_t+6,target_discharge_t+7,target_discharge_t+8,target_discharge_t+9,target_discharge_t+10,target_discharge_t+11,target_discharge_t+12,target_discharge_t+13,target_discharge_t+14,target_discharge_t+15,target_discharge_t+16,target_discharge_t+17,target_discharge_t+18,target_discharge_t+19,target_discharge_t+20,target_discharge_t+21,target_discharge_t+22,target_discharge_t+23,target_discharge_t+24
0,3,2017-01-27 22:00:00,2017-01-27 23:00:00,2017-01-28 22:00:00,124.3,-10.731833,14.042344,2017,1,27,23,4,27,0,6,104.957237,104.914683,106.543538,111.804302,112.394010,111.849558,112.712791,112.121464,108.328430,107.715725,106.889560,106.069527,107.410632,110.544313,112.394010,112.212242,112.439497,112.166844,113.215501,112.986728,113.996686,110.365441,107.890441,106.155559,105.854743,105.897666,105.768947,106.414059,106.284733,108.636046,111.713844,111.849558,110.499569,107.367116,106.673169,104.914683,104.957237,107.280134,110.142243,111.668641,110.902901,110.365441,109.874983,109.919483,109.209576,106.026537,103.351863,102.808131,102.683048,102.724726,102.599742,102.849858,102.975138,106.932889,110.320766,111.442894,108.196853,105.554752,110.276109,112.895344,113.628422,113.582470,114.504936,1

Converted 12,504 negative discharge values to missing
Converted 13,990 negative snow_depth values to missing
Converted 208 negative precipitation values to missing
Saved 32,730 rows and 207 columns to streamflow_parquet_v2/discharge_training_site_4_lb168_air24avg_precip24avg_h24.parquet

Preview for site_id=4


,site_id,history_end_time,forecast_start_time,forecast_end_time,snow_depth_latest,air_temp_avg_last_24h,precip_avg_last_24h,year,month,day,hour,day_of_week,day_of_year,season,sample_stride_hours,discharge_t-168,discharge_t-167,discharge_t-166,discharge_t-165,discharge_t-164,discharge_t-163,discharge_t-162,discharge_t-161,discharge_t-160,discharge_t-159,discharge_t-158,discharge_t-157,discharge_t-156,discharge_t-155,discharge_t-154,discharge_t-153,discharge_t-152,discharge_t-151,discharge_t-150,discharge_t-149,discharge_t-148,discharge_t-147,discharge_t-146,discharge_t-145,discharge_t-144,discharge_t-143,discharge_t-142,discharge_t-141,discharge_t-140,discharge_t-139,discharge_t-138,discharge_t-137,discharge_t-136,discharge_t-135,discharge_t-134,discharge_t-133,discharge_t-132,discharge_t-131,discharge_t-130,discharge_t-129,discharge_t-128,discharge_t-127,discharge_t-126,discharge_t-125,discharge_t-124,discharge_t-123,discharge_t-122,discharge_t-121,discharge_t-120,discharge_t-119,discharge_t-118,discharge_t-117,discharge_t-116,discharge_t-115,discharge_t-114,discharge_t-113,discharge_t-112,discharge_t-111,discharge_t-110,discharge_t-109,discharge_t-108,discharge_t-107,discharge_t-106,discharge_t-105,discharge_t-104,discharge_t-103,discharge_t-102,discharge_t-101,discharge_t-100,discharge_t-99,discharge_t-98,discharge_t-97,discharge_t-96,discharge_t-95,discharge_t-94,discharge_t-93,discharge_t-92,discharge_t-91,discharge_t-90,discharge_t-89,discharge_t-88,discharge_t-87,discharge_t-86,discharge_t-85,discharge_t-84,...,discharge_t-76,discharge_t-75,discharge_t-74,discharge_t-73,discharge_t-72,discharge_t-71,discharge_t-70,discharge_t-69,discharge_t-68,discharge_t-67,discharge_t-66,discharge_t-65,discharge_t-64,discharge_t-63,discharge_t-62,discharge_t-61,discharge_t-60,discharge_t-59,discharge_t-58,discharge_t-57,discharge_t-56,discharge_t-55,discharge_t-54,discharge_t-53,discharge_t-52,discharge_t-51,discharge_t-50,discharge_t-49,discharge_t-48,discharge_t-47,discharge_t-46,discharge_t-45,discharge_t-44,discharge_t-43,discharge_t-42,discharge_t-41,discharge_t-40,discharge_t-39,discharge_t-38,discharge_t-37,discharge_t-36,discharge_t-35,discharge_t-34,discharge_t-33,discharge_t-32,discharge_t-31,discharge_t-30,discharge_t-29,discharge_t-28,discharge_t-27,discharge_t-26,discharge_t-25,discharge_t-24,discharge_t-23,discharge_t-22,discharge_t-21,discharge_t-20,discharge_t-19,discharge_t-18,discharge_t-17,discharge_t-16,discharge_t-15,discharge_t-14,discharge_t-13,discharge_t-12,discharge_t-11,discharge_t-10,discharge_t-9,discharge_t-8,discharge_t-7,discharge_t-6,discharge_t-5,discharge_t-4,discharge_t-3,discharge_t-2,discharge_t-1,target_discharge_t+1,target_discharge_t+2,target_discharge_t+3,target_discharge_t+4,target_discharge_t+5,target_discharge_t+6,target_discharge_t+7,target_discharge_t+8,target_discharge_t+9,target_discharge_t+10,target_discharge_t+11,target_discharge_t+12,target_discharge_t+13,target_discharge_t+14,target_discharge_t+15,target_discharge_t+16,target_discharge_t+17,target_discharge_t+18,target_discharge_t+19,target_discharge_t+20,target_discharge_t+21,target_discharge_t+22,target_discharge_t+23,target_discharge_t+24
0,4,2017-01-27 22:00:00,2017-01-27 23:00:00,2017-01-28 22:00:00,124.3,-10.731833,14.042344,2017,1,27,23,4,27,0,6,217.692906,222.591091,222.955374,224.124907,225.595060,225.595060,233.084969,235.377840,234.611168,235.377840,233.084969,229.310825,227.074416,224.124907,224.858836,224.858836,230.060956,232.325428,233.084969,234.611168,235.377840,236.918366,237.692233,239.247212,233.846880,229.310825,226.333584,224.858836,224.858836,223.393264,225.595060,224.858836,226.333584,230.813428,232.325428,230.060956,226.333584,224.858836,221.429200,222.009420,221.791668,227.074416,227.817562,227.817562,227.817562,227.074416,227.074416,227.074416,224.124907,220.128994,218.335236,217.194564,218.764461,220.056974,220.705961,223.320225,223.393264,232.325428,234.611168,233.846880,233.084969,236.918366,242.386352,2

## 7. Verify Saved Parquet Files

In [15]:
for target_site_id in TARGET_SITE_IDS:
    output_path = OUTPUT_DIR / SITE_OUTPUT_TEMPLATE.format(site_id=target_site_id)
    saved_df = read_parquet_with_pyarrow(output_path)
    print(f"site_id={target_site_id} -> shape={saved_df.shape}")
    display(saved_df.iloc[:3, :20])

site_id=3 -> shape=(33108, 207)


,site_id,history_end_time,forecast_start_time,forecast_end_time,snow_depth_latest,air_temp_avg_last_24h,precip_avg_last_24h,year,month,day,hour,day_of_week,day_of_year,season,sample_stride_hours,discharge_t-168,discharge_t-167,discharge_t-166,discharge_t-165,discharge_t-164
0,3,2017-01-27 22:00:00,2017-01-27 23:00:00,2017-01-28 22:00:00,124.3,-10.731833,14.042344,2017,1,27,23,4,27,0,6,104.957237,104.914683,106.543538,111.804302,112.394010
1,3,2017-01-28 04:00:00,2017-01-28 05:00:00,2017-01-29 04:00:00,125.2,-11.605458,14.048682,2017,1,28,5,5,28,0,6,112.712791,112.121464,108.328430,107.715725,106.889560
2,3,2017-01-28 10:00:00,2017-01-28 11:00:00,2017-01-29 10:00:00,124.0,-11.267125,14.052587,2017,1,28,11,5,28,0,6,107.410632,110.544313,112.394010,112.212242,112.439497


site_id=4 -> shape=(32730, 207)


,site_id,history_end_time,forecast_start_time,forecast_end_time,snow_depth_latest,air_temp_avg_last_24h,precip_avg_last_24h,year,month,day,hour,day_of_week,day_of_year,season,sample_stride_hours,discharge_t-168,discharge_t-167,discharge_t-166,discharge_t-165,discharge_t-164
0,4,2017-01-27 22:00:00,2017-01-27 23:00:00,2017-01-28 22:00:00,124.3,-10.731833,14.042344,2017,1,27,23,4,27,0,6,217.692906,222.591091,222.955374,224.124907,225.595060
1,4,2017-01-28 04:00:00,2017-01-28 05:00:00,2017-01-29 04:00:00,125.2,-11.605458,14.048682,2017,1,28,5,5,28,0,6,233.084969,235.377840,234.611168,235.377840,233.084969
2,4,2017-01-28 10:00:00,2017-01-28 11:00:00,2017-01-29 10:00:00,124.0,-11.267125,14.052587,2017,1,28,11,5,28,0,6,227.074416,224.124907,224.858836,224.858836,230.060956


## 8. Build Air Temperature Training Parquet Files

These files forecast climate-station air temperature using only each site's own 168-hour air-temperature history plus calendar/time context. No discharge, snow, precipitation, or other hydrological/climatological support variables are included.

In [12]:
AIR_TEMPERATURE_TARGET_SITE_IDS = [1, 2]
AIR_TEMPERATURE_LOOKBACK_HOURS = 168
AIR_TEMPERATURE_FORECAST_HORIZON_HOURS = 24

AIR_TEMPERATURE_OUTPUT_DIR = Path("air_temperature_parquet_v2")
AIR_TEMPERATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AIR_TEMPERATURE_SITE_OUTPUT_TEMPLATE = "air_temperature_training_site_{site_id}_lb168_h24.parquet"

In [13]:
def build_air_temperature_hourly_table(
    engine,
    target_site_id,
    air_temp_variable_ids,
    hourly_binning="hour_end",
):
    air_temp_raw = load_raw_feature(
        engine,
        site_id=target_site_id,
        variable_ids=air_temp_variable_ids,
        feature_name="air_temp",
        allow_negative=True,
    )
    air_temp_hourly = to_hourly_last_value(air_temp_raw, "air_temp", binning=hourly_binning)

    hourly_index = pd.date_range(
        air_temp_hourly["datetime"].min(),
        air_temp_hourly["datetime"].max(),
        freq="h",
    )
    hourly = pd.DataFrame({"datetime": hourly_index})
    hourly["site_id"] = int(target_site_id)
    hourly = hourly.merge(air_temp_hourly, on="datetime", how="left")
    hourly = hourly.sort_values("datetime").reset_index(drop=True)

    hourly["air_temp_observed"] = hourly["air_temp"]
    hourly["air_temp_missing_raw"] = hourly["air_temp"].isna()

    hourly = hourly.set_index("datetime")
    hourly["air_temp"] = hourly["air_temp"].interpolate(method="time").ffill().bfill()
    hourly = hourly.reset_index()

    hourly = add_calendar_features(hourly)
    return hourly


def build_air_temperature_multihorizon_samples(
    hourly_df,
    lookback_hours=168,
    forecast_horizon_hours=24,
    max_consecutive_missing_hours=10,
):
    rows = []
    expected_step = pd.Timedelta(hours=1)
    hourly_df = hourly_df.sort_values(["site_id", "datetime"]).reset_index(drop=True)

    for site_id, site_df in hourly_df.groupby("site_id", sort=True):
        site_df = site_df.sort_values("datetime").reset_index(drop=True)
        last_start = len(site_df) - forecast_horizon_hours + 1
        target_start_idx = lookback_hours

        while target_start_idx < last_start:
            history_window = site_df.iloc[target_start_idx - lookback_hours:target_start_idx]
            future_window = site_df.iloc[target_start_idx:target_start_idx + forecast_horizon_hours]

            forecast_month = int(future_window.iloc[0]["month"])
            stride_hours = sample_stride_hours_for_month(forecast_month)

            full_window = site_df.iloc[target_start_idx - lookback_hours:target_start_idx + forecast_horizon_hours]
            full_time_window = full_window["datetime"]

            if not (full_time_window.diff().dropna() == expected_step).all():
                target_start_idx += stride_hours
                continue

            if future_window["air_temp_observed"].isna().any():
                target_start_idx += stride_hours
                continue

            if has_consecutive_missing_gap(full_window["air_temp_missing_raw"], max_consecutive_missing_hours):
                target_start_idx += stride_hours
                continue

            sample = {
                "site_id": int(site_id),
                "history_end_time": history_window.iloc[-1]["datetime"],
                "forecast_start_time": future_window.iloc[0]["datetime"],
                "forecast_end_time": future_window.iloc[-1]["datetime"],
                "year": int(future_window.iloc[0]["year"]),
                "month": int(future_window.iloc[0]["month"]),
                "day": int(future_window.iloc[0]["day"]),
                "hour": int(future_window.iloc[0]["hour"]),
                "day_of_week": int(future_window.iloc[0]["day_of_week"]),
                "day_of_year": int(future_window.iloc[0]["day_of_year"]),
                "season": int(future_window.iloc[0]["season"]),
                "sample_stride_hours": int(stride_hours),
            }

            for offset in range(lookback_hours, 0, -1):
                value = history_window.iloc[lookback_hours - offset]["air_temp"]
                sample[f"air_temp_t-{offset}"] = float(value)

            for step in range(1, forecast_horizon_hours + 1):
                value = future_window.iloc[step - 1]["air_temp_observed"]
                sample[f"target_air_temp_t+{step}"] = float(value)

            rows.append(sample)
            target_start_idx += stride_hours

    return pd.DataFrame(rows)


def build_air_temperature_training_parquet_for_site(
    engine,
    target_site_id,
    air_temp_variable_ids,
    output_path,
    lookback_hours=168,
    forecast_horizon_hours=24,
    hourly_binning="hour_end",
    max_consecutive_missing_hours=10,
):
    hourly = build_air_temperature_hourly_table(
        engine=engine,
        target_site_id=target_site_id,
        air_temp_variable_ids=air_temp_variable_ids,
        hourly_binning=hourly_binning,
    )

    training = build_air_temperature_multihorizon_samples(
        hourly,
        lookback_hours=lookback_hours,
        forecast_horizon_hours=forecast_horizon_hours,
        max_consecutive_missing_hours=max_consecutive_missing_hours,
    )

    if training.empty:
        raise ValueError(f"No air-temperature training samples created for site_id={target_site_id}")

    output_path = Path(output_path)
    write_parquet_with_pyarrow(training, output_path)
    print(f"Saved {len(training):,} rows and {training.shape[1]:,} columns to {output_path}")
    return training, hourly

In [16]:
air_temperature_outputs = {}
air_temperature_hourly_frames = {}

for target_site_id in AIR_TEMPERATURE_TARGET_SITE_IDS:
    output_path = AIR_TEMPERATURE_OUTPUT_DIR / AIR_TEMPERATURE_SITE_OUTPUT_TEMPLATE.format(site_id=target_site_id)
    training_df, hourly_df = build_air_temperature_training_parquet_for_site(
        engine=engine,
        target_site_id=target_site_id,
        air_temp_variable_ids=air_temp_variables["variable_id"].tolist(),
        output_path=output_path,
        lookback_hours=AIR_TEMPERATURE_LOOKBACK_HOURS,
        forecast_horizon_hours=AIR_TEMPERATURE_FORECAST_HORIZON_HOURS,
        hourly_binning=HOURLY_BINNING,
        max_consecutive_missing_hours=MAX_CONSECUTIVE_MISSING_HOURS,
    )
    air_temperature_outputs[target_site_id] = training_df
    air_temperature_hourly_frames[target_site_id] = hourly_df

    print(f"\nAir-temperature preview for site_id={target_site_id}")
    display(training_df.head())

Saved 35,730 rows and 204 columns to air_temperature_parquet_v2/air_temperature_training_site_1_lb168_h24.parquet

Air-temperature preview for site_id=1


,site_id,history_end_time,forecast_start_time,forecast_end_time,year,month,day,hour,day_of_week,day_of_year,season,sample_stride_hours,air_temp_t-168,air_temp_t-167,air_temp_t-166,air_temp_t-165,air_temp_t-164,air_temp_t-163,air_temp_t-162,air_temp_t-161,air_temp_t-160,air_temp_t-159,air_temp_t-158,air_temp_t-157,air_temp_t-156,air_temp_t-155,air_temp_t-154,air_temp_t-153,air_temp_t-152,air_temp_t-151,air_temp_t-150,air_temp_t-149,air_temp_t-148,air_temp_t-147,air_temp_t-146,air_temp_t-145,air_temp_t-144,air_temp_t-143,air_temp_t-142,air_temp_t-141,air_temp_t-140,air_temp_t-139,air_temp_t-138,air_temp_t-137,air_temp_t-136,air_temp_t-135,air_temp_t-134,air_temp_t-133,air_temp_t-132,air_temp_t-131,air_temp_t-130,air_temp_t-129,air_temp_t-128,air_temp_t-127,air_temp_t-126,air_temp_t-125,air_temp_t-124,air_temp_t-123,air_temp_t-122,air_temp_t-121,air_temp_t-120,air_temp_t-119,air_temp_t-118,air_temp_t-117,air_temp_t-116,air_temp_t-115,air_temp_t-114,air_temp_t-113,air_temp_t-112,air_temp_t-111,air_temp_t-110,air_temp_t-109,air_temp_t-108,air_temp_t-107,air_temp_t-106,air_temp_t-105,air_temp_t-104,air_temp_t-103,air_temp_t-102,air_temp_t-101,air_temp_t-100,air_temp_t-99,air_temp_t-98,air_temp_t-97,air_temp_t-96,air_temp_t-95,air_temp_t-94,air_temp_t-93,air_temp_t-92,air_temp_t-91,air_temp_t-90,air_temp_t-89,air_temp_t-88,air_temp_t-87,air_temp_t-86,air_temp_t-85,air_temp_t-84,air_temp_t-83,air_temp_t-82,air_temp_t-81,...,air_temp_t-76,air_temp_t-75,air_temp_t-74,air_temp_t-73,air_temp_t-72,air_temp_t-71,air_temp_t-70,air_temp_t-69,air_temp_t-68,air_temp_t-67,air_temp_t-66,air_temp_t-65,air_temp_t-64,air_temp_t-63,air_temp_t-62,air_temp_t-61,air_temp_t-60,air_temp_t-59,air_temp_t-58,air_temp_t-57,air_temp_t-56,air_temp_t-55,air_temp_t-54,air_temp_t-53,air_temp_t-52,air_temp_t-51,air_temp_t-50,air_temp_t-49,air_temp_t-48,air_temp_t-47,air_temp_t-46,air_temp_t-45,air_temp_t-44,air_temp_t-43,air_temp_t-42,air_temp_t-41,air_temp_t-40,air_temp_t-39,air_temp_t-38,air_temp_t-37,air_temp_t-36,air_temp_t-35,air_temp_t-34,air_temp_t-33,air_temp_t-32,air_temp_t-31,air_temp_t-30,air_temp_t-29,air_temp_t-28,air_temp_t-27,air_temp_t-26,air_temp_t-25,air_temp_t-24,air_temp_t-23,air_temp_t-22,air_temp_t-21,air_temp_t-20,air_temp_t-19,air_temp_t-18,air_temp_t-17,air_temp_t-16,air_temp_t-15,air_temp_t-14,air_temp_t-13,air_temp_t-12,air_temp_t-11,air_temp_t-10,air_temp_t-9,air_temp_t-8,air_temp_t-7,air_temp_t-6,air_temp_t-5,air_temp_t-4,air_temp_t-3,air_temp_t-2,air_temp_t-1,target_air_temp_t+1,target_air_temp_t+2,target_air_temp_t+3,target_air_temp_t+4,target_air_temp_t+5,target_air_temp_t+6,target_air_temp_t+7,target_air_temp_t+8,target_air_temp_t+9,target_air_temp_t+10,target_air_temp_t+11,target_air_temp_t+12,target_air_temp_t+13,target_air_temp_t+14,target_air_temp_t+15,target_air_temp_t+16,target_air_temp_t+17,target_air_temp_t+18,target_air_temp_t+19,target_air_temp_t+20,target_air_temp_t+21,target_air_temp_t+22,target_air_temp_t+23,target_air_temp_t+24
0,1,2017-02-08 19:00:00,2017-02-08 20:00:00,2017-02-09 19:00:00,2017,2,8,20,2,39,0,6,6.288,6.185,5.553,4.909,3.610,-0.103,-1.340,-2.359,-3.610,-3.977,-4.006,-4.706,-4.940,-4.943,-6.166,-4.589,-4.975,-5.997,-6.752,-7.157,-5.714,-2.296,-1.246,0.996,-0.294,0.048,0.984,1.379,1.449,1.236,0.806,0.571,0.585,-0.345,-0.531,-0.775,-0.926,-1.131,-1.323,-1.449,-1.818,-2.426,-2.837,-2.972,-2.514,-0.125,2.088,3.997,5.056,5.219,5.259,3.886,3.731,3.932,2.658,0.716,0.292,0.969,1.131,0.404,2.217,0.165,0.368,2.033,2.565,2.832,2.634,2.266,2.583,3.067,3.118,3.722,4.539,5.226,4.720,4.106,3.167,1.395,-0.301,-1.166,0.251,0.276,0.042,0.051,-0.363,0.133,-0.979,-1.546,...,0.879,2.812,4.559,5.026,5.384,5.441,5.428,4.983,4.698,2.389,0.944,0.093,-0.171,0.371,0.040,-0.407,2.421,0.524,0.266,-0.362,0.006,1.050,0.876,3.472,3.926,4.784,4.454,2.809,2.456,4.138,3.720,3.121,2.568,1.948,2.289,1.179,1.796,2.203,-0.031,-0.070,-0.223,-0.175,-0.884,-1.043,-1.206,-1.372,-1.466,-1.585,-1.568,-1.298,-0.828,1.263,2.694,3.666,5.041,5.077,5.20

Saved 35,752 rows and 204 columns to air_temperature_parquet_v2/air_temperature_training_site_2_lb168_h24.parquet

Air-temperature preview for site_id=2


,site_id,history_end_time,forecast_start_time,forecast_end_time,year,month,day,hour,day_of_week,day_of_year,season,sample_stride_hours,air_temp_t-168,air_temp_t-167,air_temp_t-166,air_temp_t-165,air_temp_t-164,air_temp_t-163,air_temp_t-162,air_temp_t-161,air_temp_t-160,air_temp_t-159,air_temp_t-158,air_temp_t-157,air_temp_t-156,air_temp_t-155,air_temp_t-154,air_temp_t-153,air_temp_t-152,air_temp_t-151,air_temp_t-150,air_temp_t-149,air_temp_t-148,air_temp_t-147,air_temp_t-146,air_temp_t-145,air_temp_t-144,air_temp_t-143,air_temp_t-142,air_temp_t-141,air_temp_t-140,air_temp_t-139,air_temp_t-138,air_temp_t-137,air_temp_t-136,air_temp_t-135,air_temp_t-134,air_temp_t-133,air_temp_t-132,air_temp_t-131,air_temp_t-130,air_temp_t-129,air_temp_t-128,air_temp_t-127,air_temp_t-126,air_temp_t-125,air_temp_t-124,air_temp_t-123,air_temp_t-122,air_temp_t-121,air_temp_t-120,air_temp_t-119,air_temp_t-118,air_temp_t-117,air_temp_t-116,air_temp_t-115,air_temp_t-114,air_temp_t-113,air_temp_t-112,air_temp_t-111,air_temp_t-110,air_temp_t-109,air_temp_t-108,air_temp_t-107,air_temp_t-106,air_temp_t-105,air_temp_t-104,air_temp_t-103,air_temp_t-102,air_temp_t-101,air_temp_t-100,air_temp_t-99,air_temp_t-98,air_temp_t-97,air_temp_t-96,air_temp_t-95,air_temp_t-94,air_temp_t-93,air_temp_t-92,air_temp_t-91,air_temp_t-90,air_temp_t-89,air_temp_t-88,air_temp_t-87,air_temp_t-86,air_temp_t-85,air_temp_t-84,air_temp_t-83,air_temp_t-82,air_temp_t-81,...,air_temp_t-76,air_temp_t-75,air_temp_t-74,air_temp_t-73,air_temp_t-72,air_temp_t-71,air_temp_t-70,air_temp_t-69,air_temp_t-68,air_temp_t-67,air_temp_t-66,air_temp_t-65,air_temp_t-64,air_temp_t-63,air_temp_t-62,air_temp_t-61,air_temp_t-60,air_temp_t-59,air_temp_t-58,air_temp_t-57,air_temp_t-56,air_temp_t-55,air_temp_t-54,air_temp_t-53,air_temp_t-52,air_temp_t-51,air_temp_t-50,air_temp_t-49,air_temp_t-48,air_temp_t-47,air_temp_t-46,air_temp_t-45,air_temp_t-44,air_temp_t-43,air_temp_t-42,air_temp_t-41,air_temp_t-40,air_temp_t-39,air_temp_t-38,air_temp_t-37,air_temp_t-36,air_temp_t-35,air_temp_t-34,air_temp_t-33,air_temp_t-32,air_temp_t-31,air_temp_t-30,air_temp_t-29,air_temp_t-28,air_temp_t-27,air_temp_t-26,air_temp_t-25,air_temp_t-24,air_temp_t-23,air_temp_t-22,air_temp_t-21,air_temp_t-20,air_temp_t-19,air_temp_t-18,air_temp_t-17,air_temp_t-16,air_temp_t-15,air_temp_t-14,air_temp_t-13,air_temp_t-12,air_temp_t-11,air_temp_t-10,air_temp_t-9,air_temp_t-8,air_temp_t-7,air_temp_t-6,air_temp_t-5,air_temp_t-4,air_temp_t-3,air_temp_t-2,air_temp_t-1,target_air_temp_t+1,target_air_temp_t+2,target_air_temp_t+3,target_air_temp_t+4,target_air_temp_t+5,target_air_temp_t+6,target_air_temp_t+7,target_air_temp_t+8,target_air_temp_t+9,target_air_temp_t+10,target_air_temp_t+11,target_air_temp_t+12,target_air_temp_t+13,target_air_temp_t+14,target_air_temp_t+15,target_air_temp_t+16,target_air_temp_t+17,target_air_temp_t+18,target_air_temp_t+19,target_air_temp_t+20,target_air_temp_t+21,target_air_temp_t+22,target_air_temp_t+23,target_air_temp_t+24
0,2,2017-01-27 22:00:00,2017-01-27 23:00:00,2017-01-28 22:00:00,2017,1,27,23,4,27,0,6,-0.814,-1.280,-1.678,-1.861,-2.068,-2.193,-2.408,-2.578,-2.432,-2.610,-2.585,-2.660,-3.028,-2.854,-3.158,-3.662,-3.540,-3.010,-2.855,-2.088,-1.559,-0.181,0.843,1.440,1.451,0.763,0.347,-0.058,-0.314,-0.449,-0.976,-1.575,0.142,-0.858,-2.633,-5.177,-3.042,-3.570,-4.687,-6.139,-7.026,-5.913,-4.362,-3.224,-2.995,-2.848,-1.511,-1.917,-0.136,-0.386,-0.609,-0.740,-0.504,-0.565,-1.294,-1.514,-1.786,-1.876,-1.792,-1.773,-1.911,-1.912,-1.895,-1.799,-1.630,-1.350,-0.486,0.200,-0.199,0.228,0.963,1.489,0.165,-0.495,-1.219,-1.314,-1.761,-2.407,-2.547,-2.548,-2.558,-2.614,-2.629,-2.661,-3.190,-3.190,-3.078,-2.929,...,-2.034,-1.713,-1.771,-1.877,-2.969,-3.215,-3.494,-3.514,-3.764,-3.966,-4.090,-4.557,-4.699,-4.894,-5.146,-6.261,-7.057,-7.617,-7.951,-7.868,-7.924,-8.080,-7.778,-7.302,-7.372,-6.692,-6.974,-6.199,-5.719,-6.119,-6.366,-6.860,-7.097,-7.420,-7.43,-7.455,-7.218,-8.010,-8.000,-8.560,-8.54,-8.650,-9.280,-8.870,-8.530,-8

In [ ]:
for target_site_id in AIR_TEMPERATURE_TARGET_SITE_IDS:
    output_path = AIR_TEMPERATURE_OUTPUT_DIR / AIR_TEMPERATURE_SITE_OUTPUT_TEMPLATE.format(site_id=target_site_id)
    saved_df = read_parquet_with_pyarrow(output_path)
    print(f"air temperature site_id={target_site_id} -> shape={saved_df.shape}")
    display(saved_df.iloc[:3, :20])